# 13buildfloods: the 2021 flood map (Stage 4, final part)

3D hexagons: **height** = people living there (as in Stages 1 to 3), **colour** = where the Sentinel-1 radar images of 20 and 22 December 2021 show a flood signal, in two tiers:

- **Strong signal:** flagged on both days.
- **Possible signal:** flagged on one day only (about one in seven of these may be a false alarm, measured on dry days).

Orange circles mark places that news reports and Wikipedia describe as flooded, so readers can see where the radar agrees and where it misses.

- Input: `hex_flood.parquet`, `flood_meta.json`, `districts.geojson`
- Output: `output/stage4_floods.html`, and for GitHub Pages `docs/floods.html`. Links to this page are added to the other three pages.

All visual settings are in Section 0, marked `# <-`.

## Section 0: settings

In [18]:
from pathlib import Path

BASE = Path.cwd()
PROCESSED = BASE / "data" / "processed"
OUTPUT = BASE / "output"
DOCS = BASE / "docs"
OUT_HTML = OUTPUT / "stage4_floods.html"

# ---- Which hexagons are in which tier (shares of the hexagon area flagged by radar) ----
STRONG_T = 0.05             # <- strong: flagged on BOTH flood days, at least this share
POSSIBLE_T = 0.05           # <- possible: flagged on at least ONE flood day, at least this share

# ---- Text ----
PAGE_TITLE = "Klang Valley Reach and Risk: the 2021 floods"                 # <- browser tab
TITLE = "Where did radar identify the 2021 floods?"                              # <- panel heading
SUBTITLE = ("Colour: where Sentinel-1 radar images from 20 and 22 December 2021 show a flood signal. "
            "Height: people living there.")                                  # <-
NOTES = [                                                                    # <- small print, one line each
    "This is a radar signal, not a full flood map. The images came 2 to 4 days after the peak (18 to 19 December), "
    "when much water had drained, and radar misses water between tall buildings and under trees. Treat it as a minimum.",
    "Strong: flagged on both days. Possible: flagged on one day only, and 178 of the 195 possible hexagons come from the 22 December image alone, so no second image confirms them. {dry_note}",
    "Each hexagon is about 0.74 km\u00b2. Even in streets known to be flooded, radar flags only part of a hexagon, "
    "so the cut-off is low ({strong_pct} of the hexagon).",
    "Orange circles: places reported flooded by news and Wikipedia; positions are approximate (town or area centre).",
    "Kontur underestimates Putrajaya's population (about 8,600 vs about 110,000 in the 2020 census).",
]
CREDIT = ("Radar: Copernicus Sentinel-1 (ESA), processed in Google Earth Engine. Ground and water layers: JRC Global Surface Water, "
          "MERIT Hydro, Copernicus DEM, ESA WorldCover 2021. Population: Kontur 2023-11-01 (CC BY 4.0). "
          "Grid: H3. Basemap: \u00a9 CARTO, \u00a9 OpenStreetMap contributors.")   # <-
LINKS = [("index.html", "See the 15-minute walking map (Stage 2)"),       # <- links at the bottom of the panel
         ("population.html", "See where people live (Stage 1)"),
         ("buses.html", "See a day of buses (Stage 3)")]
LINK_TO_THIS_PAGE = "See the 2021 floods (Stage 4)"                       # <- link added to the other pages

# ---- Basemap and camera (same as Stages 1 and 2) ----
MAP_STYLE = "road"          # <- "dark", "light" or "road"
VIEW_LAT = 3.0              # <-
VIEW_LON = 101.6            # <-
VIEW_ZOOM = 10.0            # <-
VIEW_PITCH = 40             # <-
VIEW_BEARING = -15          # <-

# ---- Hexagon colours ----
STRONG_COLOUR = "#08306b"        # <- strong signal (dark blue)
POSSIBLE_COLOUR = "#4292c6"      # <- possible signal (lighter blue)
NOSIGNAL_COLOUR = "#d0d0d0"      # <- no radar signal (this does NOT prove it stayed dry)
NOSIGNAL_OPACITY = 0.35          # <- 0 to 1, kept faint so the flood signal stands out
SIGNAL_OPACITY = 0.95            # <- opacity of strong and possible hexagons

# ---- Reported flooded places (orange circles) ----
MARKER_COLOUR = [255, 111, 0, 255]       # <- RGB + opacity
MARKER_OUTLINE = [255, 255, 255, 255]    # <-
MARKER_RADIUS_PX = 8                     # <-
REPORTED = [   # (name, longitude, latitude, what was reported)       # <- add or remove places here
    ("Taman Sri Muda, Shah Alam", 101.5345, 3.0325,
     "Whole township flooded 19 to 21 Dec; boats were the only access on 20 Dec (Malay Mail, Wikipedia)."),
    ("Dengkil", 101.6750, 2.8661,
     "One of the hardest-hit areas; over 3,500 homes reported submerged (Wikipedia)."),
    ("Meru, Klang", 101.4394, 3.1362,
     "Meru to Bukit Raja road under 2 to 3 m of water on 18 Dec (Wikipedia)."),
    ("Kuala Lumpur centre", 101.6940, 3.1540,
     "Dataran Merdeka and Jalan Masjid India reported flooded around 18 Dec (Wikipedia)."),
]

# ---- Hexagon shape (same as Stages 1 to 3) ----
ELEVATION_SCALE = 0.3       # <- metres of height per person
COVERAGE = 0.88             # <-
DISTRICT_LINE_COLOUR = [60, 60, 60, 140]     # <-

# ---- Title panel (same as Stages 1 to 3) ----
PANEL_BG = "rgba(17, 17, 20, 0.82)"          # <-
PANEL_TEXT = "#f2f2f2"                       # <-
PANEL_MUTED = "#a8a8b3"                      # <-
LINK_COLOUR = "#8ec1ff"                      # <-

# ---- Phones and tablets (lighter map) ----
PHONE_MODE = True           # <- False = phones get the same 3D map as computers
PHONE_MAX_WIDTH = 768       # <-
PHONE_3D = False            # <- False = flat hexagons on phones
PHONE_SHARP = False         # <- False = normal resolution on phones
PHONE_ZOOM = 8.6            # <-
PHONE_LAT = 3.25            # <-
PHONE_LON = 101.62          # <-
PHONE_TITLE_SIZE = 18       # <-
PHONE_PITCH = 0             # <-
PHONE_BEARING = 0           # <-
PHONE_NOTE = "On phones the map is flat and lighter. Open it on a computer for the 3D view."   # <-

In [19]:
import json, re, shutil
import numpy as np
import pandas as pd
import geopandas as gpd
import pydeck as pdk
import h3

df = pd.read_parquet(PROCESSED / "hex_flood.parquet")
meta = json.loads((PROCESSED / "flood_meta.json").read_text())
districts = gpd.read_file(PROCESSED / "districts.geojson").to_crs(4326)
print(len(df), "hexagons")

4003 hexagons


## Step 1: tiers, colours and tooltip text

In [20]:
def hex_to_rgb(code):
    code = code.lstrip("#")
    return [int(code[i:i + 2], 16) for i in (0, 2, 4)]

def pct(x):
    return f"{100 * x:.0f}%" if x >= 0.10 else f"{100 * x:.1f}%"

df["tier"] = np.where(df["flood_both"] >= STRONG_T, 2, np.where(df["flood_either"] >= POSSIBLE_T, 1, 0))
data = df[(df["tier"] > 0) | (df["population"] > 0)].copy()

TIER_NAME = {2: "Strong radar flood signal (both days)", 1: "Possible radar flood signal (one day)",
             0: "No radar flood signal (this does not mean it stayed dry)"}
COL = {2: hex_to_rgb(STRONG_COLOUR) + [int(SIGNAL_OPACITY * 255)],
       1: hex_to_rgb(POSSIBLE_COLOUR) + [int(SIGNAL_OPACITY * 255)],
       0: hex_to_rgb(NOSIGNAL_COLOUR) + [int(NOSIGNAL_OPACITY * 255)]}
data["color"] = data["tier"].map(COL)
data["title"] = (data["district"].str.replace("W.P. ", "", regex=False) + " &middot; "
                 + data["population"].map("{:,}".format) + " people")
data["body"] = data.apply(lambda r: f"<b>{TIER_NAME[r['tier']]}</b><br/>Share of this hexagon flagged: "
                          f"{pct(r['F_a'])} on 20 Dec, {pct(r['F_b'])} on 22 Dec", axis=1)
data = data[["h3", "population", "color", "title", "body"]]

# reported places: add what the radar shows in the hexagon where each one sits
sc = df.set_index("h3")
rep_rows = []
for name, lon, lat, text in REPORTED:
    c = h3.latlng_to_cell(lat, lon, 8)
    if c in sc.index:
        radar = (f"<br/>Radar in this hexagon: {pct(sc.loc[c, 'F_a'])} flagged on 20 Dec, "
                 f"{pct(sc.loc[c, 'F_b'])} on 22 Dec.")
    else:
        radar = "<br/>Outside the hexagon grid."
    rep_rows.append({"title": name, "body": text + radar, "lon": lon, "lat": lat})
reported = pd.DataFrame(rep_rows)
print(data["color"].map(tuple).value_counts().to_string())
print(reported[["title"]].to_string(index=False))

color
(208, 208, 208, 89)    2939
(66, 146, 198, 242)     195
(8, 48, 107, 242)        18
                    title
Taman Sri Muda, Shah Alam
                  Dengkil
              Meru, Klang
      Kuala Lumpur centre


## Step 2: layers, camera, tooltip, save

In [21]:
# VISUAL: 3D hexagons
hex_layer = pdk.Layer("H3HexagonLayer", data, get_hexagon="h3", get_fill_color="color", extruded=True, get_elevation="population", elevation_scale=ELEVATION_SCALE, coverage=COVERAGE, pickable=True, auto_highlight=True)
# VISUAL: district outlines
district_layer = pdk.Layer("GeoJsonLayer", json.loads(districts[["district", "geometry"]].to_json()), stroked=True, filled=False, get_line_color=DISTRICT_LINE_COLOUR, line_width_min_pixels=1)
# VISUAL: reported flooded places (drawn on top of the columns)
marker_layer = pdk.Layer("ScatterplotLayer", reported, get_position=["lon", "lat"], get_fill_color=MARKER_COLOUR, get_line_color=MARKER_OUTLINE, stroked=True, line_width_min_pixels=2, radius_min_pixels=MARKER_RADIUS_PX, radius_max_pixels=MARKER_RADIUS_PX + 4, get_radius=300, pickable=True, parameters={"depthTest": False})

view = pdk.ViewState(latitude=VIEW_LAT, longitude=VIEW_LON, zoom=VIEW_ZOOM, pitch=VIEW_PITCH, bearing=VIEW_BEARING)

# VISUAL: tooltip
tooltip = {"html": "<b>{title}</b><br/>{body}",
           "style": {"backgroundColor": "#1b1b22", "color": "#f2f2f2", "fontSize": "13px", "maxWidth": "340px",
                     "fontFamily": "Segoe UI, Arial, sans-serif", "padding": "8px 10px", "lineHeight": "1.5"}}

OUTPUT.mkdir(exist_ok=True)
deck = pdk.Deck(layers=[district_layer, hex_layer, marker_layer], initial_view_state=view, map_provider="carto", map_style=MAP_STYLE, tooltip=tooltip)
deck.to_html(str(OUT_HTML), notebook_display=False)
print("Saved", OUT_HTML.name)

Saved stage4_floods.html


## Step 3: title panel with legend and key figures

In [22]:
def cal(t):
    for r in meta["calibration"]:
        if abs(r["threshold"] - t) < 1e-9:
            return r
    return None

total_pop = int(df["population"].sum())
n = {t: int((df["tier"] == t).sum()) for t in (0, 1, 2)}
ppl = {t: int(df.loc[df["tier"] == t, "population"].sum()) for t in (0, 1, 2)}
def about(x):
    return f"about {int(round(x, -3)):,}" if x >= 1000 else f"{x:,}"

cs, cp = cal(STRONG_T), cal(POSSIBLE_T)
if cs and cp:
    dry_note = (f"On two dry days (8 and 9 December) the same method flagged {cp['dry_either']} hexagons as possible "
                f"and {cs['dry_both']} as strong, against {cp['flood_either']} and {cs['flood_both']} on the flood days.")
else:
    dry_note = "The same method flagged far fewer hexagons on two dry days in early December."
notes = "".join('<div class="small">' + t.format(dry_note=dry_note, strong_pct=pct(STRONG_T)) + '</div>' for t in NOTES)

rows = [
    (STRONG_COLOUR, 1.0, "Strong signal (both days)", f"{n[2]} hexagons, {about(ppl[2])} people"),
    (POSSIBLE_COLOUR, 1.0, "Possible signal (one day)", f"{n[1]} hexagons, {about(ppl[1])} people"),
    (NOSIGNAL_COLOUR, NOSIGNAL_OPACITY, "No radar signal", "does not mean it stayed dry"),
]
legend_rows = "".join(
    f'<div class="row"><span class="sw" style="background:{c};opacity:{max(o, 0.5)}"></span>'
    f'<span class="lab">{lab}</span><span class="pct">{txt}</span></div>' for c, o, lab, txt in rows)
mark_row = (f'<div class="row"><span class="sw dot" style="background:rgb({MARKER_COLOUR[0]},{MARKER_COLOUR[1]},{MARKER_COLOUR[2]})"></span>'
            f'<span class="lab">Reported flooded</span><span class="pct">place from news reports</span></div>')

def panel_html():
    links = "".join(f'<div class="lnk" style="font-size:12px; margin-top:6px;"><a href="{h}">{t}</a></div>' for h, t in LINKS)
    return f'''
<style>
  #kr-panel {{ position:absolute; top:12px; left:12px; z-index:10; max-width:min(400px, calc(100vw - 48px));
    background:{PANEL_BG}; color:{PANEL_TEXT}; padding:14px 16px; border-radius:8px;
    font-family:"Segoe UI", Arial, sans-serif; font-size:13px; line-height:1.45; }}
  #kr-panel h1 {{ font-size:24px; margin:0 0 4px; font-weight:600; }}
  #kr-panel .sub {{ color:{PANEL_MUTED}; margin-bottom:10px; }}
  #kr-panel .row {{ display:flex; align-items:center; gap:8px; margin:3px 0; }}
  #kr-panel .sw {{ width:14px; height:14px; border-radius:3px; flex:none; }}
  #kr-panel .sw.dot {{ border-radius:50%; border:2px solid #fff; box-sizing:border-box; }}
  #kr-panel .lab {{ width:120px; flex:none; }}
  #kr-panel .pct {{ color:{PANEL_MUTED}; }}
  #kr-panel .facts {{ margin-top:10px; }}
  #kr-panel .small {{ color:{PANEL_MUTED}; font-size:11px; margin-top:6px; }}
  #kr-panel a {{ color:{LINK_COLOUR}; }}
  @media (max-width: 600px) {{ #kr-panel {{ font-size:12px; padding:10px 12px; }} #kr-panel .small, #kr-panel .sub {{ display:none; }} #kr-panel .lab {{ width:100px; }} }}
</style>
<div id="kr-panel">
  <h1>{TITLE}</h1>
  <div class="sub">{SUBTITLE}</div>
  {legend_rows}
  {mark_row}
  <div class="facts">Radar flagged {n[2] + n[1]} of {len(df):,} hexagons, where {about(ppl[2] + ppl[1])} people live ({100 * (ppl[2] + ppl[1]) / total_pop:.0f}% of the Klang Valley). Not every flagged hexagon was flooded, and some flooded places are missed.</div>
  {notes}
  <div class="small">{CREDIT}</div>
  {links}
</div>
'''

html = OUT_HTML.read_text(encoding="utf-8")
html = html.replace("<title>pydeck</title>", f"<title>{PAGE_TITLE}</title>", 1)
html = html.replace("</body>", panel_html() + "\n</body>", 1)
OUT_HTML.write_text(html, encoding="utf-8")
print("Saved", OUT_HTML.name, "|", round(OUT_HTML.stat().st_size / 1024), "KB")

Saved stage4_floods.html | 1359 KB


## Step 3b: a lighter map on phones

Same phone switch as the Stage 2 page: flat hexagons, normal resolution, zoomed out. Computers get the full 3D map.

In [23]:
def add_phone_mode(html):
    """Insert the phone switch into a pydeck page (only once)."""
    if not PHONE_MODE or "kr-phone-mode" in html:
        return html
    marker = "const deckInstance = createDeck("
    assert marker in html, "pydeck page layout changed: phone switch not added"
    script = f'''// kr-phone-mode: lighter map on phones and tablets
    const krPhone = window.matchMedia("(max-width: {PHONE_MAX_WIDTH}px)").matches
                    || window.matchMedia("(pointer: coarse)").matches;
    if (krPhone) {{
      const v = jsonInput.initialViewState;
      v.zoom = {PHONE_ZOOM}; v.pitch = {PHONE_PITCH}; v.bearing = {PHONE_BEARING};
      v.latitude = {PHONE_LAT}; v.longitude = {PHONE_LON};
      (jsonInput.layers || []).forEach(l => {{
        if (l["@@type"] === "H3HexagonLayer") {{ l.extruded = {str(PHONE_3D).lower()}; l.autoHighlight = false; }}
      }});
      jsonInput.useDevicePixels = {str(PHONE_SHARP).lower()};
      document.documentElement.classList.add("kr-phone");
    }}
    '''
    html = html.replace(marker, script + marker, 1)
    note = (f'<div class="kr-phone-note" style="font-size:11px; margin-top:8px; opacity:0.8;">{PHONE_NOTE}</div>')
    css = ("<style>.kr-phone-note {display:none;} html.kr-phone .kr-phone-note {display:block;} "
           f"html.kr-phone #kr-panel h1 {{font-size:{PHONE_TITLE_SIZE}px;}}</style>")
    html = html.replace('<div class="lnk"', note + "\n  " + '<div class="lnk"', 1)
    viewport = '<meta name="viewport" content="width=device-width, initial-scale=1">'
    if 'name="viewport"' not in html:
        css = viewport + "\n" + css
    html = html.replace("</head>", css + "\n</head>", 1)
    return html

OUT_HTML.write_text(add_phone_mode(OUT_HTML.read_text(encoding="utf-8")), encoding="utf-8")
print("Phone mode added to", OUT_HTML.name, "|", round(OUT_HTML.stat().st_size / 1024), "KB")

Phone mode added to stage4_floods.html | 1360 KB


## Step 4: GitHub Pages files and links

`docs/floods.html` is this map. A link to it is added to `docs/index.html`, `docs/population.html` and `docs/buses.html`. Re-running 06buildaccess or 09buildbuses rewrites those pages without the link, so re-run this step afterwards.

In [24]:
DOCS.mkdir(exist_ok=True)
shutil.copyfile(OUT_HTML, DOCS / "floods.html")

link = (f'<div class="lnk" style="font-size:12px; margin-top:8px;">'
        f'<a href="floods.html" style="color:{LINK_COLOUR}">{LINK_TO_THIS_PAGE}</a></div>')
for name in ["index.html", "population.html"]:
    f = DOCS / name
    if not f.exists():
        print(name, "not found"); continue
    s = f.read_text(encoding="utf-8")
    if 'href="floods.html"' in s:
        print(name, "already links to floods.html"); continue
    i = s.rfind('<div class="lnk"')
    if i < 0:
        print(name, "no link block found, not changed"); continue
    j = s.find("</div>", i) + len("</div>")
    f.write_text(s[:j] + "\n  " + link + s[j:], encoding="utf-8")
    print(name, "link added")

# buses.html builds its links from a list inside the page
f = DOCS / "buses.html"
if f.exists():
    s = f.read_text(encoding="utf-8")
    old = '["population.html", "See where people live (Stage 1)"]'
    if "floods.html" in s:
        print("buses.html already links to floods.html")
    elif old in s:
        f.write_text(s.replace(old, old + ', ["floods.html", ' + json.dumps(LINK_TO_THIS_PAGE) + "]", 1), encoding="utf-8")
        print("buses.html link added")
    else:
        print("buses.html: link list not found. Add the floods link to LINKS in 09buildbuses Section 0 and re-run 09, then this step.")

for f in sorted(DOCS.glob("*.html")):
    print(f.name, round(f.stat().st_size / 1024), "KB")

index.html already links to floods.html
population.html already links to floods.html
buses.html already links to floods.html
buses.html 21 KB
floods.html 1360 KB
index.html 1596 KB
population.html 1016 KB


In [25]:
import webbrowser
webbrowser.open((DOCS / "floods.html").resolve().as_uri())

True

___
**END**